In [ ]:
import os
import glob
import hashlib
import json
import time
import subprocess
import sys
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ================= CONFIGURATION =================
# 1. INSERT YOUR VIRUSTOTAL API KEY HERE
VT_API_KEY = '**REMOVED FOR PRIVACY REASONS**' 

# 2. Input Directory (Where your Ransomware binaries are)
BINARY_DIR = '/mnt/mydrive/Datasets/Nithish Dataset/Sorted_Dataset/Files/Ransomware'

# 3. Output Directory (Where we will save the generated JSON reports)
JSON_OUTPUT_DIR = 'vt_reports_ransomware'

# 4. AVClass Output File
AVCLASS_RESULTS = 'avclass_families.txt'
# =================================================

def calculate_sha256(filepath):
    """Generates the SHA256 hash required to query VirusTotal."""
    sha256 = hashlib.sha256()
    try:
        with open(filepath, 'rb') as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception as e:
        print(f"[-] Could not hash {filepath}: {e}")
        return None

def get_vt_report(file_hash):
    """Queries VirusTotal API to create the JSON report."""
    url = f"https://www.virustotal.com/api/v3/files/{file_hash}"
    headers = {"x-apikey": VT_API_KEY}
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            print("    [!] API Quota Exceeded. Pausing for 65 seconds...")
            time.sleep(65)
            return get_vt_report(file_hash) # Retry
        elif response.status_code == 404:
            print(f"    [-] Hash not found in VirusTotal database: {file_hash}")
            return None
        else:
            print(f"    [!] Error {response.status_code}: {response.text}")
            return None
    except Exception as e:
        print(f"    [!] Connection error: {e}")
        return None

def process_avclass(json_folder, output_file):
    """Runs the AVClass tool on the folder of generated JSONs."""
    print(f"\n[*] Running AVClass labeling on {json_folder}...")
    
    # Check if we have JSONs
    if not glob.glob(os.path.join(json_folder, "*.json")):
        print("[-] No JSON files found to analyze. AVClass skipped.")
        return False

    try:
        # Command line equivalent: avclass -d vt_reports -o output.txt -t 2 -hash sha256
        cmd = [
            sys.executable, '-m', 'avclass.tool',
            '-d', json_folder,
            '-o', output_file,
            '-t', '2',       # Threshold: 2 AVs must agree
            '-hash', 'sha256'
        ]
        
        # Run AVClass
        subprocess.run(cmd, check=True)
        print(f"[+] AVClass labeling finished. Results saved to: {output_file}")
        return True
    except subprocess.CalledProcessError:
        print("[-] AVClass failed. Make sure it is installed (pip install avclass).")
        return False
    except Exception as e:
        print(f"[-] Unexpected error running AVClass: {e}")
        return False

def visualize_results(label_file):
    """Reads AVClass results and creates a chart."""
    if not os.path.exists(label_file):
        return

    print("\n[*] Generating distribution chart...")
    try:
        # Load results (Format: sha256 \t family \t alias_info)
        # Using python engine to handle potential irregularities
        df = pd.read_csv(label_file, sep='\t', names=['Hash', 'Family', 'Info'], engine='python', header=None)
        
        # Filter noise
        df = df[~df['Family'].isin(['SINGLETON', 'NULL'])]
        
        if df.empty:
            print("[-] No valid families identified (all were SINGLETON or NULL).")
            return

        # Top 15 Families
        top_families = df['Family'].value_counts().head(15)
        
        # Plot
        plt.figure(figsize=(12, 8))
        sns.barplot(x=top_families.values, y=top_families.index, palette='rocket')
        
        plt.title('Top Ransomware Families (Identified via AVClass)', fontsize=16)
        plt.xlabel('Number of Samples', fontsize=12)
        plt.ylabel('Family Name', fontsize=12)
        
        # Add counts
        for i, v in enumerate(top_families.values):
            plt.text(v + 0.5, i, str(v), color='black', va='center')

        plt.tight_layout()
        plt.savefig('ransomware_families_chart.png', dpi=300)
        print("[+] Chart saved: ransomware_families_chart.png")
        
    except Exception as e:
        print(f"[-] Visualization error: {e}")

def main():
    if not os.path.exists(BINARY_DIR):
        print(f"Error: Directory not found: {BINARY_DIR}")
        return

    # Create output folder for JSONs
    os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)
    
    # List all files
    binaries = [f for f in glob.glob(os.path.join(BINARY_DIR, '*')) if os.path.isfile(f)]
    total_files = len(binaries)
    
    print(f"[*] Found {total_files} binaries in source folder.")
    print(f"[*] Starting Scan (Hash -> VirusTotal Query -> Save JSON)...")
    print("-" * 60)

    processed_count = 0
    
    for binary_path in binaries:
        filename = os.path.basename(binary_path)
        
        # 1. Calculate Hash
        file_hash = calculate_sha256(binary_path)
        if not file_hash:
            continue
            
        # Check if we already have this report locally
        json_path = os.path.join(JSON_OUTPUT_DIR, f"{file_hash}.json")
        if os.path.exists(json_path):
            processed_count += 1
            continue
            
        # 2. Query VirusTotal
        print(f"    Scanning: {filename} ({file_hash[:8]}...)")
        report = get_vt_report(file_hash)
        
        if report:
            # 3. Save JSON
            with open(json_path, 'w') as f:
                json.dump(report, f, indent=4)
        
        processed_count += 1
        
        # Rate Limiting check (Free API is strict)
        # We handle this inside get_vt_report mostly, but adding a small safety buffer helps
        # time.sleep(15) # Uncomment if using a very restrictive free key

    print("-" * 60)
    print(f"[*] Downloaded/Verified {processed_count} JSON reports.")
    
    # 4. Run AVClass to identify families
    if process_avclass(JSON_OUTPUT_DIR, AVCLASS_RESULTS):
        # 5. Visualize
        visualize_results(AVCLASS_RESULTS)

if __name__ == "__main__":
    main()

[*] Found 15448 binaries in source folder.
[*] Starting Scan (Hash -> VirusTotal Query -> Save JSON)...
------------------------------------------------------------
    Scanning: 0003c4ac06a2ae23eff788a3ff179174f588fbf28e876621c64943aa5db072e8 (0003c4ac...)
    Scanning: 0007eb3ccafe69715a2184404083637512e9c93b1b44949534f1a6fc4cf01d5f (0007eb3c...)
    Scanning: 000a0b2e04e49119a62a1ec860e88e97374864789f9b51d561376d412e85bd4e (000a0b2e...)
    Scanning: 000dca0ac54c86a00cadcbec0af08128508603bc303049f79cc41f89f678f6b1 (000dca0a...)
    Scanning: 00131a827e218ca3505cd5e3cf5201889d6930b0c8bbcf31d6414842931ab3dd (00131a82...)
    Scanning: 00133805d692da064e8e47b1d06298998764c5284606bbcd79ef753ca68cac41 (00133805...)
    Scanning: 001c8cb8cf7e63cd6ed5f7ce49ad4e94fca8da7783abc0caac86b33f3c18f3a4 (001c8cb8...)
    Scanning: 001d6a2019f88fd6e27fb625888b97478a492248a8c48dc35b3f0c77becf2b98 (001d6a20...)
    Scanning: 001dd7be6ede435578ae4668310bf8a7f937ec23790c00e49fc6c6ce7f01e290 (001dd7be...

/home/garvitagarwal/anaconda3/envs/pyenv/bin/python: No module named avclass.tool


In [6]:
import os
import glob
import json
import subprocess
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ================= CONFIGURATION =================
# 1. Source Directory (Your individual VT JSONs)
JSON_SOURCE_DIR = 'vt_reports_ransomware_old/'

# 2. Intermediate Merged File (Created by this script)
MERGED_INPUT_FILE = 'merged_reports.jsonl'

# 3. Euphony Output Directory
EUPHONY_OUTPUT_DIR = 'euphony_results'

# 4. Path to JAR
EUPHONY_JAR_PATH = './euphony.jar' 
# =================================================

def merge_json_files(source_dir, output_file):
    """
    Reads all individual JSON files and writes them to a single file 
    where each line is one valid JSON object (JSON Lines format).
    """
    print(f"[*] Merging JSON files from {source_dir}...")
    
    files = glob.glob(os.path.join(source_dir, "*.json"))
    if not files:
        print("[-] No JSON files found to merge.")
        return False

    with open(output_file, 'w') as outfile:
        count = 0
        for fpath in files:
            try:
                with open(fpath, 'r') as infile:
                    data = json.load(infile)
                    # Write as a single line (flattened)
                    outfile.write(json.dumps(data) + "\n")
                    count += 1
                    if count % 1000 == 0:
                        print(f"    Merged {count} files...")
            except Exception as e:
                print(f"    [!] Failed to merge {fpath}: {e}")
                
    print(f"[+] Successfully merged {count} reports into {output_file}")
    return True

def run_euphony(input_file, output_dir, jar_path):
    """Runs Euphony with the correct -r (file) and -e (dir) flags."""
    print(f"\n[*] Running Euphony classification...")
    
    # Create output dir if not exists
    os.makedirs(output_dir, exist_ok=True)

    # Command: java -jar euphony.jar -r merged.jsonl -e results_dir -D
    # -D (--export-cluster-mapping) ensures we get the Hash -> Family mapping
    cmd = [
        'java', '-jar', jar_path,
        '-r', input_file,     # Input (Single merged file)
        '-e', output_dir,     # Output Directory
        '-D'                  # Export clustering results
    ]

    print(f"    Command: {' '.join(cmd)}")
    
    try:
        subprocess.run(cmd, check=True)
        print(f"[+] Euphony finished successfully.")
        return True
    except subprocess.CalledProcessError as e:
        print(f"[-] Euphony failed with error code {e.returncode}.")
        return False

def visualize_results(output_dir):
    """Finds the result CSV in the output dir and plots it."""
    print(f"\n[*] Parsing results in {output_dir}...")
    
    # Euphony usually outputs a file named 'clusters.csv' or similar in the export dir
    # We look for any CSV created there.
    results_files = glob.glob(os.path.join(output_dir, "*.csv"))
    
    if not results_files:
        print("[-] No CSV results found in output directory. Checking for JSON...")
        results_files = glob.glob(os.path.join(output_dir, "*.json"))
        
    if not results_files:
        print("[-] No results file found to visualize.")
        return

    # Pick the most likely file (usually the largest one or specifically named 'clusters')
    target_file = results_files[0] 
    print(f"    Reading {target_file}...")

    try:
        # Try loading. Euphony CSVs are often headerless: "label, count" or "hash, label"
        # We will inspect the first few lines to guess structure
        df = pd.read_csv(target_file, header=None)
        
        # Heuristic: If column 0 is long (hash) and column 1 is string (family)
        if len(df.columns) >= 2:
            # Rename for clarity
            df.columns = ['Hash', 'Family'] + [f'Col_{i}' for i in range(2, len(df.columns))]
        else:
            print("[-] CSV format unexpected. Could not auto-parse columns.")
            return

        # Filter noise
        df = df[~df['Family'].str.lower().isin(['singleton', 'unknown', 'null'])]

        # Top 15
        top_families = df['Family'].value_counts().head(15)

        # Plot
        plt.figure(figsize=(12, 8))
        sns.barplot(x=top_families.values, y=top_families.index, palette='magma')
        plt.title('Top Ransomware Families (Euphony Results)', fontsize=16)
        plt.xlabel('Count', fontsize=12)
        
        for i, v in enumerate(top_families.values):
            plt.text(v + 0.5, i, str(v), color='black', va='center')

        plt.tight_layout()
        plt.savefig('euphony_final_distribution.png', dpi=300)
        print("[+] Visualization saved: euphony_final_distribution.png")

    except Exception as e:
        print(f"[-] Visualization error: {e}")

if __name__ == "__main__":
    # Step 1: Merge
    if merge_json_files(JSON_SOURCE_DIR, MERGED_INPUT_FILE):
        # Step 2: Run
        if run_euphony(MERGED_INPUT_FILE, EUPHONY_OUTPUT_DIR, EUPHONY_JAR_PATH):
            # Step 3: Plot
            visualize_results(EUPHONY_OUTPUT_DIR)

[*] converting v3 JSONs from vt_reports_ransomware to v2 format...
    Converted 1000 reports...
    Converted 2000 reports...
    Converted 3000 reports...
    Converted 4000 reports...
    Converted 5000 reports...
    Converted 6000 reports...
    Converted 7000 reports...
    Converted 8000 reports...
    Converted 9000 reports...
    Converted 10000 reports...
    Converted 11000 reports...
    Converted 12000 reports...
    Converted 13000 reports...
    Converted 14000 reports...
    Converted 15000 reports...
[+] Done! Saved 15398 compatible reports to euphony_compatible.jsonl
    Now run Euphony on 'euphony_compatible.jsonl'


In [9]:
import json
import glob
import os
import datetime

# ================= CONFIGURATION =================
# Directory containing your original V3 JSONs
INPUT_DIR = 'vt_reports_ransomware'

# The output file we will feed into Euphony
OUTPUT_FILE = 'euphony_data.jsonl'
# =================================================

def convert_to_legacy_v2(v3_data):
    """
    Converts a modern VirusTotal v3 JSON into the strict legacy v2 format
    required by Euphony (specifically adding the 'resource' key).
    """
    try:
        # 1. Extract Attributes (Handle different V3 dump formats)
        data = v3_data.get('data', {})
        attr = data.get('attributes', {})
        if not attr and 'last_analysis_results' in v3_data:
             # Handle case where user saved just the attributes
            attr = v3_data

        # 2. Extract Mandatory Identifiers
        # Euphony CRASHES if 'resource' is missing. It uses this as the ID.
        sha256 = data.get('id', attr.get('sha256', ''))
        if not sha256: return None

        # 3. Format Date (Legacy tools often expect strings, not timestamps)
        epoch = attr.get('last_analysis_date', 0)
        try:
            date_str = datetime.datetime.utcfromtimestamp(epoch).strftime('%Y-%m-%d %H:%M:%S')
        except:
            date_str = "2026-01-01 00:00:00"

        # 4. Build 'scans' Dictionary
        # V3 format: {"McAfee": {"category": "malicious", "result": "WannaCry"}}
        # V2 format: {"McAfee": {"detected": true, "result": "WannaCry"}}
        v3_scans = attr.get('last_analysis_results', {})
        v2_scans = {}

        for vendor, res in v3_scans.items():
            # Only include if it actually detected something
            if res.get('category') == 'malicious' and res.get('result'):
                v2_scans[vendor] = {
                    "detected": True,
                    "result": res['result'],
                    "version": res.get('engine_version', "1.0"),
                    "update": res.get('engine_update', "20220101")
                }

        # SAFETY CHECK: Euphony throws "not-empty labels" assertion if a file has 0 labels.
        # We MUST skip files that are clean or have no AV detections.
        if not v2_scans:
            return None

        # 5. Construct Final Legacy Object
        return {
            "response_code": 1,
            "verbose_msg": "Scan finished, information embedded",
            "resource": sha256,     # <--- CRITICAL FIX: Euphony looks for this specific key
            "scan_id": f"{sha256}-{epoch}",
            "sha256": sha256,
            "scan_date": date_str,
            "positives": len(v2_scans),
            "total": len(v3_scans),
            "scans": v2_scans
        }

    except Exception as e:
        return None

def main():
    print(f"[*] Reading V3 JSONs from '{INPUT_DIR}'...")
    files = glob.glob(os.path.join(INPUT_DIR, "*.json"))
    
    valid_count = 0
    skipped_count = 0
    
    with open(OUTPUT_FILE, 'w') as outfile:
        for fpath in files:
            try:
                with open(fpath, 'r') as infile:
                    content = json.load(infile)
                
                legacy_entry = convert_to_legacy_v2(content)
                
                if legacy_entry:
                    # Write as JSON Lines (one JSON object per line)
                    outfile.write(json.dumps(legacy_entry) + "\n")
                    valid_count += 1
                else:
                    skipped_count += 1
                    
            except Exception:
                skipped_count += 1
    
    print("-" * 50)
    print(f"[+] SUCCESS: Converted {valid_count} reports.")
    print(f"[-] SKIPPED: {skipped_count} reports (clean/empty/error).")
    print(f"[*] Output saved to: {OUTPUT_FILE}")
    print("-" * 50)
    print("Run Euphony now with:")
    print(f"java -jar euphony.jar -r {OUTPUT_FILE} -e euphony_results -D")

if __name__ == "__main__":
    main()

[*] Reading V3 JSONs from 'vt_reports_ransomware'...


/tmp/ipykernel_72612/245358998.py:35: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_str = datetime.datetime.utcfromtimestamp(epoch).strftime('%Y-%m-%d %H:%M:%S')


--------------------------------------------------
[+] SUCCESS: Converted 15398 reports.
[-] SKIPPED: 50 reports (clean/empty/error).
[*] Output saved to: euphony_data.jsonl
--------------------------------------------------
Run Euphony now with:
java -jar euphony.jar -r euphony_data.jsonl -e euphony_results -D


In [7]:
java -jar euphony.jar -r euphony_compatible.jsonl -e euphony_results -D

SyntaxError: invalid syntax (3550664782.py, line 1)